Prepare analogy stimuli for the -er inflection experiment.

In [ ]:
from collections import Counter, defaultdict
import functools
import pickle

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from src.analysis.state_space import StateSpaceAnalysisSpec
from src.analysis import analogy

In [ ]:
train_dataset = "librispeech-train-clean-100"
dataset = train_dataset
state_space_specs_path = f"outputs/state_space_specs/librispeech-train-clean-100/w2v2_8/state_space_specs.h5"
base_model = "w2v2_pc_8"

base_model_class = base_model.rsplit("_", maxsplit=1)[0]

pos_counts_path = "data/pos_counts.pkl"
output_dir = f"outputs/analogy_v3/inputs/{dataset}/{base_model_class}"

seed = 1234

min_samples_per_word = 5
max_samples_per_word = 100

In [ ]:
state_space_spec = StateSpaceAnalysisSpec.from_hdf5(state_space_specs_path, "word")
state_space_spec = state_space_spec.subsample_instances(max_samples_per_word)

In [ ]:
with open(pos_counts_path, "rb") as f:
    pos_counts = pickle.load(f)

In [ ]:
cuts_df = state_space_spec.cuts.xs("phoneme", level="level").drop(columns=["onset_frame_idx", "offset_frame_idx"])
cuts_df["label_idx"] = cuts_df.index.get_level_values("label").map({l: i for i, l in enumerate(state_space_spec.labels)})
cuts_df["frame_idx"] = cuts_df.groupby(["label", "instance_idx"]).cumcount()
cuts_df = cuts_df.reset_index().set_index(["label", "instance_idx", "frame_idx"]).sort_index()

In [ ]:
cut_phonemic_forms = cuts_df.groupby(["label", "instance_idx"]).description.agg(' '.join)

In [ ]:
ss_spans = state_space_spec.target_frame_spans_df

In [ ]:
label_to_idx = {l: i for i, l in enumerate(state_space_spec.labels)}

In [ ]:
er_df = pd.read_csv("data/er_forms.csv")
critical_columns = ["agentive", "comparative", "nonmorphemic"]

# drop rows missing annotations
er_df = er_df.loc[~er_df[critical_columns].isna().all(axis=1)]

# drop rows labeled "drop"
er_df = er_df.loc[~er_df[critical_columns].isin(["drop"]).any(axis=1)]

# convert to boolean
assert {np.nan, "y"} == set(er_df[critical_columns].values.flatten())
er_df[critical_columns] = er_df[critical_columns].notna()

# ignore cases where multiple inflections are possible
er_df = er_df.loc[er_df[critical_columns].sum(axis=1) == 1]

# expand where there are multiple base forms, comma-separated
er_df = er_df.assign(
    base=er_df.bases.str.split(",")
).explode("base").drop(columns=["bases"]).reset_index(drop=True)

er_df["inflected_phones"] = er_df.description
er_df["base_phones"] = er_df.inflected_phones.str.rsplit(" ER", n=1).str[0]
er_df["inflected"] = er_df.label
er_df = er_df.drop(columns=["description", "label"])

er_df.loc[er_df.agentive, "inflection"] = "agentive"
er_df.loc[er_df.comparative, "inflection"] = "comparative"
er_df.loc[er_df.nonmorphemic, "inflection"] = "nonmorphemic"
er_df = er_df.drop(columns=critical_columns)

In [ ]:
er_df.head()

In [ ]:
# retrieve base instances
inflection_instances = pd.merge(er_df, cut_phonemic_forms.reset_index(),
         left_on=["base", "base_phones"],
         right_on=["label", "description"]) \
    .rename(columns={"instance_idx": "base_instance_idx"}).drop(columns=["label", "description"])

# only retain instances with sufficient samples
inflection_instances = inflection_instances.groupby("base").filter(
    lambda df: len(df) >= min_samples_per_word
)

In [ ]:
# now outer-join with inflected instances
inflection_instances = pd.merge(inflection_instances,
         cut_phonemic_forms.reset_index(),
         left_on=["inflected", "inflected_phones"],
         right_on=["label", "description"]) \
    .rename(columns={"instance_idx": "inflected_instance_idx"}).drop(columns=["label", "description"])

# only retain inflected instances with sufficient samples
inflection_instances = inflection_instances.groupby("inflected").filter(
    lambda df: df.inflected_instance_idx.nunique() >= min_samples_per_word
)

In [ ]:
inflection_instances["base_idx"] = inflection_instances.base.map(label_to_idx)
inflection_instances["inflected_idx"] = inflection_instances.inflected.map(label_to_idx)

## Set up random stimuli

In [ ]:
labels = state_space_spec.label_counts
labels = set(labels[labels > min_samples_per_word].index)

In [ ]:
# Add on random word pair baseline
num_random_word_pairs = inflection_instances.groupby("inflection").inflected.nunique().max()
random_word_pairs = np.random.choice(len(list(labels)), size=(num_random_word_pairs, 2))
random_word_pairs = pd.DataFrame(random_word_pairs, columns=["base_idx", "inflected_idx"])
random_word_pairs["base"] = random_word_pairs.base_idx.map({i: l for i, l in enumerate(state_space_spec.labels)})
random_word_pairs["inflected"] = random_word_pairs.inflected_idx.map({i: l for i, l in enumerate(state_space_spec.labels)})
random_word_pairs["is_regular"] = False
random_word_pairs["inflection"] = "random"
random_word_pairs = random_word_pairs.set_index("inflection")
random_word_pairs

In [ ]:
# Generate cross-instances for random pairs
random_instances = pd.merge(random_word_pairs.reset_index(),
         cut_phonemic_forms.reset_index(),
         left_on=["base"],
         right_on=["label"]).rename(columns={"instance_idx": "base_instance_idx", "description": "base_phones"}).drop(columns=["label"])
random_instances = pd.merge(random_instances,
            cut_phonemic_forms.reset_index(),
            left_on=["inflected"],
            right_on=["label"]).rename(columns={"instance_idx": "inflected_instance_idx", "description": "inflected_phones"}).drop(columns=["label"])
random_instances

In [ ]:
inflection_instances = pd.concat([inflection_instances, random_instances])

## Save

In [ ]:
state_space_spec.to_hdf5(f"{output_dir}/state_space_spec.h5")

In [ ]:
inflection_instances.to_parquet(f"{output_dir}/inflection_instances.parquet")